# Merge the workshop annotations

Builds one dataset per data domain, then a single combined dataset, from

    ../Annotations/<domain>/<person_folder>/{classes.json, train/, val/, test/}

This notebook lives in `notebooks/`, so every path is relative to the repo root
one level up. `../Annotations/` is treated as read-only: cell 1 copies it to
`../Annotations_processed/` and every later step reads and writes there.

The five merged datasets land under `../data/`:

    ../data/{earth, energy, life, materials, combined}

Each has the same shape as an individual export: `classes.json` plus `train/`,
`val/` and `test/` holding `images/` + `masks/`.

Prefixes keep everything separable. Files become `<person>_<filename>` and class
labels become `<person>_<classname>`, so nothing collides and no two people's
classes are ever fused — even when they use the same word for different things
("void" means something different in every one of these datasets).

In [1]:
# ============ 1. SETTINGS + make sure every person has train/ and val/ ============
from pathlib import Path
import json
import shutil

import numpy as np
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

# This notebook sits in notebooks/, so everything is one level up.
ROOT = Path("..")
ANNOTATIONS = ROOT / "Annotations"            # read-only source: <domain>/<person>/...
PROCESSED = ROOT / "Annotations_processed"    # working copy of the above
DATA = ROOT / "data"                          # all merged datasets land here
DOMAINS = {                                   # folder on disk -> output name
    "Earth Sciences":    "earth",
    "Energy Science":    "energy",
    "Life Sciences":     "life",
    "Material Sciences": "materials",
}
COMBINED = DATA / "combined"

IGNORE = 255                          # unannotated pixels, in every export
SPLITS = ["train", "val", "test"]
EXTS = {".png", ".tif", ".tiff", ".jpg", ".jpeg"}

# The full person-folder name is used as the prefix: it is guaranteed unique,
# whereas a shortened name would collide (Marr_Dataset1 / Marr_Dataset1_better).
# Add entries here to shorten a prefix by hand, e.g. {"Song_Shale_dataset_1": "Song"}.
RENAME = {}

prefix_of = lambda p: RENAME.get(p.name, p.name)
is_image = lambda p: p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith(".")
read_mask = lambda p: (lambda a: a[..., 0] if a.ndim == 3 else a)(np.asarray(Image.open(p)))
load_classes = lambda root: {int(k): v for k, v in json.loads((root / "classes.json").read_text()).items()}

people = lambda domain: sorted(p for p in domain.iterdir() if p.is_dir() and (p / "classes.json").is_file())


# --- work on a copy so the original Annotations/ is never modified ----------
if PROCESSED.exists():
    shutil.rmtree(PROCESSED)
shutil.copytree(ANNOTATIONS, PROCESSED)
print(f"copied {ANNOTATIONS}/ -> {PROCESSED}/\n")

# --- training needs both a train/ and a val/ split; fill in whichever is
# --- missing by duplicating the other one
print("checking splits")
fixed = 0
for domain_dir in [PROCESSED / d for d in DOMAINS]:
    for person in people(domain_dir):
        has = {s for s in ("train", "val") if (person / s / "images").is_dir()}
        if has == {"train", "val"}:
            continue
        if not has:
            print(f"  SKIP  {domain_dir.name}/{person.name}  (neither train/ nor val/)")
            continue
        src, dst = ("val", "train") if "val" in has else ("train", "val")
        print(f"  no {dst}/ -> copying {src}/   {domain_dir.name}/{person.name}")
        shutil.copytree(person / src, person / dst)
        fixed += 1

print(f"\n{fixed} folder(s) given a missing split")

copied ../Annotations/ -> ../Annotations_processed/

checking splits
  no train/ -> copying val/   Earth Sciences/Dechert_Dataset1_Try2__141833_76B_1_ming_x00y00_00000_20260804_230312
  no train/ -> copying val/   Earth Sciences/MCELHANY_DATASET1
  no train/ -> copying val/   Earth Sciences/Manga__150503_U1593B1H4W_B_10X_00160_20260804_230315
  no train/ -> copying val/   Life Sciences/Caetano__61015_220835_CS_EFM_3_22080700_20260804_223728
  no train/ -> copying val/   Life Sciences/McCurry_pistach_stem_control
  no train/ -> copying val/   Life Sciences/Sangjan_Plant_Leaf
  no train/ -> copying val/   Life Sciences/lydia__browse_dataset_MBC D10000_20260804_224441
  no train/ -> copying val/   Material Sciences/Eli_Hiss__browse_dataset_3_20260804_233339
  no train/ -> copying val/   Material Sciences/Feru_Dataset1
  no train/ -> copying val/   Material Sciences/Marr_Dataset1_better
  no train/ -> copying val/   Material Sciences/Pokharel__browse_1B_20260804_232040

11 folder(s) given 

In [2]:
# ============ 2. MERGE EACH DOMAIN'S PEOPLE INTO ONE FOLDER ============
def merge(sources, out, quiet=False):
    """Combine dataset folders into `out`, prefixing files and labels by folder name.

    Class ids are reassigned into one global space; mask pixels are remapped
    through a per-source lookup table. 255 stays 255 in and out.
    """
    names = [prefix_of(s) for s in sources]
    if len(set(names)) != len(names):
        raise ValueError(f"duplicate prefixes in {names} -- outputs would overwrite each other")

    # -- build the global class table and one LUT per source
    global_classes, luts, next_id = {}, [], 0
    for src, name in zip(sources, names):
        classes = load_classes(src)
        lut = np.full(256, IGNORE, dtype=np.uint8)      # anything unmapped -> ignore
        for local_id in sorted(classes):
            if local_id == IGNORE:
                continue                                 # ignore folds straight through
            if not 0 <= local_id < IGNORE:
                raise ValueError(f"{src}: class id {local_id} out of range")
            if next_id >= IGNORE:
                raise ValueError(f"more than {IGNORE} classes -- would collide with ignore")
            # labels are copied verbatim apart from the prefix, so a source that
            # declares the same label twice keeps both, with separate ids
            global_classes[next_id] = f"{name}_{classes[local_id]}"
            lut[local_id] = next_id
            next_id += 1
        luts.append(lut)

    # -- copy images, remap masks
    totals = {s: 0 for s in SPLITS}
    for src, name, lut in zip(sources, names, luts):
        declared = set(load_classes(src)) | {IGNORE}
        for split in SPLITS:
            img_dir, mask_dir = src / split / "images", src / split / "masks"
            if not img_dir.is_dir():
                continue
            for img in sorted(p for p in img_dir.iterdir() if is_image(p)):
                mask_path = mask_dir / img.name
                if not is_image(mask_path):
                    print(f"    WARNING: no mask for {src.name}/{split}/{img.name}, skipping")
                    continue
                mask = read_mask(mask_path)
                unknown = sorted(set(np.unique(mask).tolist()) - declared)
                if unknown:
                    print(f"    WARNING: {src.name}/{split}/{img.name} has undeclared {unknown} -> ignore")

                (out / split / "images").mkdir(parents=True, exist_ok=True)
                (out / split / "masks").mkdir(parents=True, exist_ok=True)
                new_name = f"{name}_{img.name}"
                shutil.copy2(img, out / split / "images" / new_name)
                # always .png -- a lossy .jpg mask would invent class ids
                Image.fromarray(lut[mask]).save((out / split / "masks" / new_name).with_suffix(".png"))
                totals[split] += 1

    out.mkdir(parents=True, exist_ok=True)
    # 255 is the ignore value and is deliberately absent from classes.json
    (out / "classes.json").write_text(json.dumps(
        {str(k): v for k, v in global_classes.items()}, indent=2))

    counts = ", ".join(f"{s}={totals[s]}" for s in SPLITS if totals[s])
    print(f"  {out}  <- {len(sources)} sources, {len(global_classes)} classes, {counts}")
    if not quiet:
        for gid, label in global_classes.items():
            print(f"      {gid:>3}: {label}")
    return global_classes


DATA.mkdir(parents=True, exist_ok=True)

print("merging people into domain datasets\n")
for domain, out_name in DOMAINS.items():
    src = PROCESSED / domain
    out = DATA / out_name
    if out.exists():
        shutil.rmtree(out)          # rebuild from scratch so reruns are clean
    print(f"{domain}:")
    merge(people(src), out, quiet=True)
    print()

merging people into domain datasets

Earth Sciences:
  ../data/earth  <- 4 sources, 19 classes, train=11, val=6

Energy Science:
  ../data/energy  <- 4 sources, 15 classes, train=21, val=6

Life Sciences:
  ../data/life  <- 5 sources, 19 classes, train=12, val=5, test=1

Material Sciences:
  ../data/materials  <- 4 sources, 13 classes, train=4, val=4



In [3]:
# ============ 3. MERGE THE FOUR DOMAINS INTO ONE COMBINED DATASET ============
if COMBINED.exists():
    shutil.rmtree(COMBINED)

print("merging domains into the combined dataset\n")
merge([DATA / n for n in DOMAINS.values()], COMBINED)

print(f"\nnow run from the repo root:  python finetune.py data/{COMBINED.name}")

merging domains into the combined dataset

  ../data/combined  <- 4 sources, 66 classes, train=48, val=21, test=1
        0: earth_Dechert_Dataset1_Try2__141833_76B_1_ming_x00y00_00000_20260804_230312_glass
        1: earth_Dechert_Dataset1_Try2__141833_76B_1_ming_x00y00_00000_20260804_230312_min (dark,plag)
        2: earth_Dechert_Dataset1_Try2__141833_76B_1_ming_x00y00_00000_20260804_230312_pore
        3: earth_Dechert_Dataset1_Try2__141833_76B_1_ming_x00y00_00000_20260804_230312_min (medium,opx)
        4: earth_Dechert_Dataset1_Try2__141833_76B_1_ming_x00y00_00000_20260804_230312_min (bright,ox)
        5: earth_MCELHANY_DATASET1_pore
        6: earth_MCELHANY_DATASET1_air
        7: earth_MCELHANY_DATASET1_void
        8: earth_MCELHANY_DATASET1_unhydrated cement
        9: earth_MCELHANY_DATASET1_hydrated cement paste
       10: earth_MCELHANY_DATASET1_fiber
       11: earth_Manga__150503_U1593B1H4W_B_10X_00160_20260804_230315_gas
       12: earth_Manga__150503_U1593B1H4W_B_10X